# FakeNewsNet — Complete Fake News Detection Pipeline
**Dataset**: FakeNewsNet (PolitiFact + GossipCop) · 23,196 articles  
**Models**: Logistic Regression · Linear SVM · Naive Bayes · Bidirectional LSTM  
**Tasks**: Binary classification (Fake vs Real)  
---

## 1. Install & Import Libraries

In [ ]:
# Install dependencies
!pip install scikit-learn pandas numpy matplotlib seaborn tensorflow --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, sys, csv, json, os, re
warnings.filterwarnings('ignore')
csv.field_size_limit(sys.maxsize)

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                              classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder
import joblib

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='darkgrid')
print("All libraries imported successfully")

## 2. Load & Explore Dataset

In [ ]:
# Load all 4 CSV files from the FakeNewsNet dataset
BASE = '../dataset'   # adjust path if needed

dfs = []
for source in ['politifact', 'gossipcop']:
    for label in ['fake', 'real']:
        path = f'{BASE}/{source}_{label}.csv'
        df_temp = pd.read_csv(path, encoding='utf-8')
        df_temp['label'] = label
        df_temp['source'] = source
        dfs.append(df_temp)
        print(f"  Loaded {source}_{label}: {len(df_temp)} rows")

df = pd.concat(dfs, ignore_index=True)
df = df[['id','title','news_url','label','source']].dropna(subset=['title'])
df['title'] = df['title'].astype(str).str.strip()
df = df[df['title'].str.len() > 5]
df['label_binary'] = (df['label'] == 'fake').astype(int)

print(f"\nTotal dataset: {len(df):,} articles")
df.head()

In [ ]:
print("=== Dataset Summary ===")
print(df['label'].value_counts())
print()
print("=== Source Distribution ===")
print(df['source'].value_counts())
print()
print("=== Class Balance ===")
print(df['label'].value_counts(normalize=True).round(3))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#00c896', '#ff4d6d']

# Class distribution
counts = df['label'].value_counts()
axes[0].bar(['Real','Fake'], [counts['real'], counts['fake']],
            color=colors, width=0.5, edgecolor='white', linewidth=1.5)
for i, v in enumerate([counts['real'], counts['fake']]):
    axes[0].text(i, v+200, f'{v:,}', ha='center', fontweight='bold', fontsize=13)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Source breakdown
src = df.groupby(['source','label']).size().unstack()
src.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white', width=0.6)
axes[1].set_title('Source vs Label', fontsize=14, fontweight='bold')
axes[1].set_xticklabels(['GossipCop','PolitiFact'], rotation=0)
axes[1].legend(['Real','Fake'])
axes[1].grid(axis='y', alpha=0.3)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

# Title length distribution
df['word_count'] = df['title'].str.split().str.len()
for lbl, c, n in [('real','#00c896','Real'), ('fake','#ff4d6d','Fake')]:
    axes[2].hist(df[df['label']==lbl]['word_count'], bins=35,
                 alpha=0.7, color=c, label=n, edgecolor='none')
axes[2].set_title('Headline Word Count Distribution', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Word Count')
axes[2].set_ylabel('Frequency')
axes[2].legend()
axes[2].grid(alpha=0.3)
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.suptitle('FakeNewsNet — Exploratory Data Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Average word count
print("Average headline word count:")
print(df.groupby('label')['word_count'].describe().round(2))

## 4. Text Preprocessing

In [ ]:
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'http\S+', '', text)          # remove URLs
    text = re.sub(r'[^a-z0-9\s]', ' ', text)    # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()    # normalise whitespace
    return text

df['title_clean'] = df['title'].apply(clean_text)
print("Sample cleaned titles:")
for _, row in df.sample(5, random_state=42).iterrows():
    print(f"  [{row['label'].upper()}] {row['title_clean']}")

## 5. Train / Test Split

In [ ]:
X = df['title_clean']
y = df['label_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"\nLabel balance in train:")
print(y_train.value_counts(normalize=True).round(3))

## 6. Scikit-learn Model Training

### 6.1 Define Pipelines (TF-IDF + Classifier)

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1,2),
                                   stop_words='english', sublinear_tf=True)),
        ('clf',   LogisticRegression(max_iter=1000, C=1.0, random_state=42))
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1,2),
                                   stop_words='english', sublinear_tf=True)),
        ('clf',   LinearSVC(max_iter=2000, C=1.0, random_state=42))
    ]),
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1,2),
                                   stop_words='english')),
        ('clf',   MultinomialNB(alpha=0.1))
    ]),
}
print("Models defined:", list(models.keys()))

### 6.2 Train and Evaluate All Models

In [ ]:
results = {}

for name, pipeline in models.items():
    print(f"Training {name}...")
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    prec = precision_score(y_test, y_pred, average='weighted')
    rec  = recall_score(y_test, y_pred, average='weighted')

    results[name] = {
        'pipeline': pipeline,
        'y_pred': y_pred,
        'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec
    }
    print(f"  Accuracy : {acc*100:.2f}%")
    print(f"  F1 Score : {f1:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print()

best_name = max(results, key=lambda k: results[k]['f1'])
print(f"Best model: {best_name} (F1={results[best_name]['f1']:.4f})")

### 6.3 Classification Reports

In [ ]:
for name, res in results.items():
    print(f"=== {name} ===")
    print(classification_report(y_test, res['y_pred'], target_names=['Real','Fake']))
    print()

### 6.4 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Real','Fake'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAcc={res["accuracy"]*100:.2f}%', fontweight='bold')
plt.suptitle('Confusion Matrices — Test Set', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.5 Model Comparison Plot

In [ ]:
names = list(results.keys())
accs  = [results[n]['accuracy']*100 for n in names]
f1s   = [results[n]['f1']           for n in names]
precs = [results[n]['precision']     for n in names]
recs  = [results[n]['recall']        for n in names]
palette = ['#4d9fff','#f5d800','#00c896']

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
metrics = [('Accuracy (%)', accs), ('F1 Score', f1s),
           ('Precision', precs), ('Recall', recs)]
ymins  = [80, 0.80, 0.80, 0.80]

for ax, (metric, vals), ymin in zip(axes, metrics, ymins):
    bars = ax.bar(names, vals, color=palette, width=0.5, edgecolor='white', linewidth=1.5)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, v+(0.3 if 'Acc' in metric else 0.003),
                f'{v:.2f}' + ('%' if 'Acc' in metric else ''),
                ha='center', fontweight='bold', fontsize=11)
    ax.set_title(metric, fontweight='bold', fontsize=13)
    ax.set_ylim(ymin, 100 if 'Acc' in metric else 1.0)
    ax.tick_params(axis='x', rotation=12)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Scikit-learn Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.6 Feature Importance (Top Words)

In [ ]:
best_pipe = results[best_name]['pipeline']
tfidf      = best_pipe.named_steps['tfidf']
clf        = best_pipe.named_steps['clf']
feat_names = np.array(tfidf.get_feature_names_out())

if hasattr(clf, 'coef_'):
    coefs     = clf.coef_.flatten()
    top_fake  = np.argsort(coefs)[-20:][::-1]
    top_real  = np.argsort(coefs)[:20]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].barh(feat_names[top_fake][::-1], coefs[top_fake][::-1], color='#ff4d6d', edgecolor='none')
    axes[0].set_title('Top 20 FAKE News Word Features', fontweight='bold', fontsize=13)
    axes[0].set_xlabel('Coefficient Weight')
    axes[0].grid(axis='x', alpha=0.3)

    axes[1].barh(feat_names[top_real][::-1], np.abs(coefs[top_real][::-1]), color='#00c896', edgecolor='none')
    axes[1].set_title('Top 20 REAL News Word Features', fontweight='bold', fontsize=13)
    axes[1].set_xlabel('|Coefficient Weight|')
    axes[1].grid(axis='x', alpha=0.3)

    plt.suptitle(f'Most Important TF-IDF Features — {best_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

### 6.7 Save Best Scikit-learn Model

In [ ]:
os.makedirs('models', exist_ok=True)
for name, res in results.items():
    joblib.dump(res['pipeline'], f"models/{name.replace(' ','_')}.pkl")
    print(f"Saved: models/{name.replace(' ','_')}.pkl")

joblib.dump(results[best_name]['pipeline'], 'models/best_model.pkl')
print(f"\nBest model saved: models/best_model.pkl ({best_name})")

## 7. Deep Learning — Bidirectional LSTM

### 7.1 Tokenise Text for LSTM

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

MAX_WORDS = 15000
MAX_LEN   = 30
EMBED_DIM = 64

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_tr_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_te_seq = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding='post')

print(f"Vocabulary size : {len(tokenizer.word_index):,}")
print(f"Sequence shape  : {X_tr_seq.shape}")

### 7.2 Build Bidirectional LSTM Model

In [ ]:
def build_bilstm():
    inp = Input(shape=(MAX_LEN,), name='input')
    x   = layers.Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN, name='embedding')(inp)
    x   = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.3), name='bilstm_1')(x)
    x   = layers.Bidirectional(layers.LSTM(32, dropout=0.3), name='bilstm_2')(x)
    x   = layers.Dense(64, activation='relu', name='dense_1')(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation='sigmoid', name='output')(x)
    model = Model(inp, out, name='BiLSTM_FakeNews')
    model.compile(optimizer=Adam(0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

lstm_model = build_bilstm()
lstm_model.summary()

### 7.3 Train LSTM with Early Stopping

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1)
]

history = lstm_model.fit(
    X_tr_seq, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

### 7.4 LSTM Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     'o-', color='#4d9fff', label='Train', lw=2)
axes[0].plot(history.history['val_accuracy'], 's--',color='#f5d800', label='Validation', lw=2)
axes[0].set_title('LSTM — Accuracy per Epoch', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     'o-', color='#ff4d6d', label='Train', lw=2)
axes[1].plot(history.history['val_loss'], 's--',color='#00c896', label='Validation', lw=2)
axes[1].set_title('LSTM — Loss per Epoch', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Bidirectional LSTM Training History', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('lstm_training.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.5 Evaluate LSTM

In [ ]:
y_prob_lstm = lstm_model.predict(X_te_seq, verbose=0).flatten()
y_pred_lstm = (y_prob_lstm >= 0.5).astype(int)

acc_lstm = accuracy_score(y_test, y_pred_lstm)
f1_lstm  = f1_score(y_test, y_pred_lstm, average='weighted')
pr_lstm  = precision_score(y_test, y_pred_lstm, average='weighted')
re_lstm  = recall_score(y_test, y_pred_lstm, average='weighted')

print("=== Bidirectional LSTM Results ===")
print(f"  Accuracy  : {acc_lstm*100:.2f}%")
print(f"  F1 Score  : {f1_lstm:.4f}")
print(f"  Precision : {pr_lstm:.4f}")
print(f"  Recall    : {re_lstm:.4f}")
print()
print(classification_report(y_test, y_pred_lstm, target_names=['Real','Fake']))

### 7.6 Save LSTM Model

In [ ]:
import pickle
lstm_model.save('models/lstm_model.h5')
pickle.dump(tokenizer, open('models/tokenizer.pkl','wb'))
print("LSTM model and tokenizer saved.")

## 8. Final Comparison — All Models

In [ ]:
all_results = {
    **{name: {'accuracy': res['accuracy'], 'f1': res['f1'],
              'precision': res['precision'], 'recall': res['recall']}
       for name, res in results.items()},
    'Bi-LSTM': {'accuracy': acc_lstm, 'f1': f1_lstm,
                'precision': pr_lstm, 'recall': re_lstm}
}

print(f"{'Model':<25} {'Accuracy':>10} {'F1':>10} {'Precision':>10} {'Recall':>10}")
print("-"*70)
for name, m in sorted(all_results.items(), key=lambda x: -x[1]['f1']):
    print(f"{name:<25} {m['accuracy']*100:>9.2f}% {m['f1']:>10.4f} {m['precision']:>10.4f} {m['recall']:>10.4f}")

In [ ]:
names  = list(all_results.keys())
accs   = [all_results[n]['accuracy']*100 for n in names]
f1s    = [all_results[n]['f1'] for n in names]
colors = ['#4d9fff','#f5d800','#00c896','#ff4d6d']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
b1 = axes[0].bar(names, accs, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
for b,v in zip(b1,accs):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.2, f'{v:.2f}%', ha='center', fontweight='bold')
axes[0].set_title('All Models — Accuracy', fontweight='bold', fontsize=13)
axes[0].set_ylim(75, 100); axes[0].tick_params(axis='x', rotation=12)
axes[0].grid(axis='y', alpha=0.3); axes[0].spines['top'].set_visible(False)

b2 = axes[1].bar(names, f1s, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
for b,v in zip(b2,f1s):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.003, f'{v:.4f}', ha='center', fontweight='bold')
axes[1].set_title('All Models — F1 Score', fontweight='bold', fontsize=13)
axes[1].set_ylim(0.75, 1.0); axes[1].tick_params(axis='x', rotation=12)
axes[1].grid(axis='y', alpha=0.3); axes[1].spines['top'].set_visible(False)

plt.suptitle('Complete Model Comparison (ML + Deep Learning)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('full_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBest Overall Model: {max(all_results, key=lambda k: all_results[k]['f1'])}")

## 9. Conclusions

This notebook demonstrated a complete fake news detection pipeline on the **FakeNewsNet dataset**:

**Dataset**
- 23,196 articles from PolitiFact and GossipCop
- 24.8% fake, 75.2% real (imbalanced dataset)
- Features: news headline text only

**Models Compared**

| Model | Type | Strength |
|---|---|---|
| Logistic Regression | Classical ML | Fast, interpretable, good baseline |
| Linear SVM | Classical ML | Strong with text data, reliable |
| Naive Bayes | Classical ML | Very fast, good with TF-IDF |
| Bi-LSTM | Deep Learning | Captures word order and context |

**Key Findings**
- TF-IDF + Linear SVM achieves strong results on headline text alone
- Bidirectional LSTM captures sequential patterns in text
- Feature importance shows sensational words strongly indicate fake news
- Cross-source generalisation (PolitiFact vs GossipCop) is a key challenge

**Future Work**
- Use full article body text (not just headlines)
- Fine-tune BERT/RoBERTa for higher accuracy
- Add social context features (tweet count, user profiles)